In [2]:
import os

os.chdir(os.path.join(
    os.path.expanduser("~"),
    "OneDrive",
    "Documents",
    "Water_Usage_Analysis"
))

print("Current folder:", os.getcwd())
print("Contents:", os.listdir())

Current folder: C:\Users\Tushar\OneDrive\Documents\Water_Usage_Analysis
Contents: ['25_water_consumption.csv', 'Water_Usage_Analysis.ipynb']


In [ ]:
# PROJECT : WATER USAGE & CONSERVATION ANALYSIS
# Libraries: Pandas and NumPy
# Name: Tushar
# Branch: ElCE-2
# Admission No: 2026B02310111
#GitHub Link: https://github.com/ttushar007/water-usage-analysis

import pandas as pd
import numpy as np


# TASK 1: LOAD AND INSPECT THE DATASET

df = pd.read_csv("25_water_consumption.csv")

print("\nTASK 1: DATASET INSPECTION")
print("Dataset Shape (Rows, Columns):", df.shape)

print("\nColumn Names:")
print(list(df.columns))

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Records:")
print(df.head())


# TASK 2: IDENTIFY MISSING VALUES AND DUPLICATES

print("\nTASK 2: DATA QUALITY CHECK")

print("\nMissing Values per Column:")
print(df.isnull().sum())

duplicate_count = df["Meter_Record_ID"].duplicated().sum()
print("\nDuplicate Meter_Record_ID Entries:", duplicate_count)

# Remove duplicate records and keep the first occurrence
df = df.drop_duplicates(subset=["Meter_Record_ID"], keep="first").copy()

print("\nShape After Removing Duplicates:", df.shape)


# TASK 3: HANDLE MISSING WATER USAGE VALUES

print("\nTASK 3: MISSING VALUE HANDLING")

missing_before = df["Water_Usage_Liters"].isnull().sum()
print("Missing Water_Usage_Liters Before:", missing_before)

# Calculate overall median as a backup value
overall_median = df["Water_Usage_Liters"].median()

# Fill missing values using the median of each Household_Type
household_types = df["Household_Type"].dropna().unique()

for h_type in household_types:
    group_median = df[df["Household_Type"] == h_type]["Water_Usage_Liters"].median()
    
    # If a group has no valid numbers at all, use the overall median
    if pd.isnull(group_median):
        group_median = overall_median
        
    condition = (df["Household_Type"] == h_type) & (df["Water_Usage_Liters"].isnull())
    df.loc[condition, "Water_Usage_Liters"] = group_median

# Fill any remaining missing values (if Household_Type itself was missing)
df["Water_Usage_Liters"] = df["Water_Usage_Liters"].fillna(overall_median)

missing_after = df["Water_Usage_Liters"].isnull().sum()
print("Missing Water_Usage_Liters After:", missing_after)

print("\nMethod Explanation:")
print("Missing water usage values were filled with the median of their Household_Type.")
print("Median was chosen because water usage has extreme outliers that would skew the mean.")


# TASK 4: DATETIME CONVERSION AND EXTRACTION

print("\nTASK 4: DATE ANALYSIS")

df["Reading_Date"] = pd.to_datetime(df["Reading_Date"], errors="coerce")

print("Invalid or Missing Dates:", df["Reading_Date"].isnull().sum())

# Extract Year, Month, and Weekday using simple datetime attributes
df["Year"] = df["Reading_Date"].dt.year
df["Month"] = df["Reading_Date"].dt.month
df["Weekday"] = df["Reading_Date"].dt.day_name()

print("\nDate Extraction Preview:")
print(df[["Reading_Date", "Year", "Month", "Weekday"]].head())


# TASK 5: TOTAL AND AVERAGE WATER CONSUMPTION

print("\nTASK 5: WATER CONSUMPTION")

total_consumption = df["Water_Usage_Liters"].sum()

# Total and average consumption per household
household_total = df.groupby("Household_ID")["Water_Usage_Liters"].sum()
household_avg = df.groupby("Household_ID")["Water_Usage_Liters"].mean()

household_summary = pd.DataFrame({
    "Total_Consumption": household_total,
    "Average_Consumption": household_avg
})

average_per_household = household_total.mean()

print("Total Water Consumption:", round(total_consumption, 2), "Liters")
print("Average Total Consumption Per Household:", round(average_per_household, 2), "Liters")

print("\nHousehold Comparison Table (First 5):")
print(household_summary.head())


# TASK 6: MONTH-TO-MONTH CONSUMPTION CHANGE

print("\nTASK 6: MONTHLY HOUSEHOLD CHANGE")

valid_dates_df = df.dropna(subset=["Year", "Month"]).copy()

monthly_household = (
    valid_dates_df.groupby(["Household_ID", "Year", "Month"])["Water_Usage_Liters"]
    .sum()
    .reset_index()
    .sort_values(["Household_ID", "Year", "Month"])
)

# Calculate month-to-month difference for each household
monthly_household["Consumption_Change"] = (
    monthly_household.groupby("Household_ID")["Water_Usage_Liters"].diff()
)

# Check if the months are truly consecutive (no skipped months)
month_index = monthly_household["Year"] * 12 + monthly_household["Month"]
month_gap = month_index.groupby(monthly_household["Household_ID"]).diff()

# If a month was skipped (gap > 1), set change to NaN so it doesn't mislead
monthly_household.loc[month_gap != 1, "Consumption_Change"] = np.nan

print(monthly_household.head(15))


# TASK 7: RANK HOUSEHOLD TYPES BY TOTAL WATER USAGE

print("\nTASK 7: HOUSEHOLD TYPE RANKING")

household_type_usage = (
    df.groupby("Household_Type")["Water_Usage_Liters"]
    .sum()
    .sort_values(ascending=False)
)

print(household_type_usage)


# TASK 8: COMPARE AVERAGE USAGE ACROSS BUILDING TYPES

print("\nTASK 8: BUILDING TYPE COMPARISON")

building_comparison = (
    df.groupby("Building_Type")["Water_Usage_Liters"]
    .mean()
    .sort_values(ascending=False)
)

print("Average Water Consumption by Building Type:")
print(building_comparison)


# TASK 9: TOP 10 HIGHEST-CONSUMING HOUSEHOLDS

print("\nTASK 9: TOP 10 HOUSEHOLDS")

top_10_households = household_summary.sort_values(
    by="Total_Consumption",
    ascending=False
).head(10)

print(top_10_households)


# TASK 10: MONTH WITH HIGHEST WATER USAGE

print("\nTASK 10: MONTHLY WATER USAGE TREND")

monthly_usage = (
    valid_dates_df.groupby(["Year", "Month"])["Water_Usage_Liters"]
    .sum()
    .sort_values(ascending=False)
)

if len(monthly_usage) > 0:
    highest_year_month = monthly_usage.index[0]
    highest_usage = monthly_usage.iloc[0]

    print("Month with Highest Consumption (Year, Month):", highest_year_month)
    print("Highest Monthly Consumption:", round(highest_usage, 2), "Liters")

    print("\nMonthly Water-Use Trend (Chronological):")
    print(monthly_usage.sort_index())
else:
    print("No valid monthly data available.")


# TASK 11: DICTIONARY AND SET

print("\nTASK 11: DICTIONARY AND SET")

zone_usage = (
    df.groupby("Zone")["Water_Usage_Liters"]
    .sum()
    .sort_values(ascending=False)
)

# Store zone-wise consumption totals in a dictionary
zone_dictionary = zone_usage.to_dict()

print("Zone-wise Consumption Dictionary:")
print(zone_dictionary)

# Formatted Zone-wise Consumption Table (Expected Deliverable)
zone_table = zone_usage.reset_index()
zone_table.columns = ["Zone", "Total_Water_Usage_Liters"]
print("\nZone-wise Consumption Table:")
print(zone_table)

# Store unique zones in a set
unique_zones = set(df["Zone"].dropna())

print("\nUnique Zones Set:")
print(unique_zones)
print("Number of Unique Zones:", len(unique_zones))


# TASK 12: CONSUMPTION CLASSIFICATION FUNCTION

print("\nTASK 12: CONSUMPTION CLASSIFICATION")

q1 = df["Water_Usage_Liters"].quantile(0.25)
q3 = df["Water_Usage_Liters"].quantile(0.75)

def classify_consumption(usage):
    if usage < q1:
        return "Low"
    elif usage <= q3:
        return "Normal"
    else:
        return "High"

df["Consumption_Level"] = df["Water_Usage_Liters"].apply(classify_consumption)

print("Low Threshold (< Q1):", round(q1, 2), "Liters")
print("High Threshold (> Q3):", round(q3, 2), "Liters")

print("\nConsumption Classification Counts:")
print(df["Consumption_Level"].value_counts())


# TASK 13: NUMPY STATISTICAL ANALYSIS

print("\nTASK 13: NUMPY STATISTICAL SUMMARY")

water_array = df["Water_Usage_Liters"].to_numpy()

mean_usage = np.mean(water_array)
median_usage = np.median(water_array)
variance_usage = np.var(water_array)
std_usage = np.std(water_array)

print("Mean:", round(mean_usage, 2))
print("Median:", round(median_usage, 2))
print("Variance:", round(variance_usage, 2))
print("Standard Deviation:", round(std_usage, 2))


# TASK 14: IQR OUTLIER DETECTION AND COMPARISON

print("\nTASK 14: IQR HIGH-CONSUMPTION OUTLIER ANALYSIS")

Q1 = np.percentile(water_array, 25)
Q3 = np.percentile(water_array, 75)
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

print("Q1:", round(Q1, 2))
print("Q3:", round(Q3, 2))
print("IQR:", round(IQR, 2))
print("Upper Limit (High Consumption Cutoff):", round(upper_limit, 2))

# Filter strictly for unusually HIGH consumption records
high_outliers = df[df["Water_Usage_Liters"] > upper_limit]

print("\nUnusually High Consumption Records:")
print(high_outliers[["Meter_Record_ID", "Household_ID", "Household_Type", "Zone", "Water_Usage_Liters"]])
print("\nNumber of Unusually High Records:", len(high_outliers))

# Compare average usage across household types
average_by_household = (
    df.groupby("Household_Type")["Water_Usage_Liters"]
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage Consumption by Household Type:")
print(average_by_household)


# TASK 15: WATER METER CLASS AND CONSERVATION RECOMMENDATIONS

print("\nTASK 15: WATER METER CLASS")

class WaterMeter:
    def __init__(self, meter_id, household_type):
        self.meter_id = meter_id
        self.household_type = household_type

    def display_details(self):
        print("Meter ID:", self.meter_id)
        print("Household Type:", self.household_type)

# Create an object using the first record in the dataset
first_record = df.iloc[0]
meter1 = WaterMeter(first_record["Meter_Record_ID"], first_record["Household_Type"])
meter1.display_details()


print("\nFIVE WATER CONSERVATION RECOMMENDATIONS")

highest_zone = zone_usage.index[0]
highest_building = building_comparison.index[0]
highest_household_type = average_by_household.index[0]
highest_household_id = top_10_households.index[0]

recommendations = [
    f"1. Conduct water audits in {highest_zone}, as it recorded the highest total water consumption.",
    f"2. Install low-flow fixtures in {highest_building} buildings, which showed the highest average usage per reading.",
    f"3. Run awareness campaigns for {highest_household_type} households, which have the highest average consumption among household types.",
    f"4. Inspect meter and plumbing for Household {highest_household_id} (highest overall consumer) and any flagged IQR outlier households for leaks.",
    "5. Introduce tiered water pricing during peak consumption months and promote rainwater harvesting."
]

for rec in recommendations:
    print(rec)

    output_path = "25_water_consumption_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"\nCleaned dataset saved to: {output_path}")

print("\nPROJECT COMPLETED")